In [1]:
import sys; print(sys.executable)
import tensorflow as tf
from dense_pcn_layer import DensePCNLayer
from conv_pcn_layer import Conv2DPCNLayer
from transformer_pcn_layer import TransformerPCNLayer, PositionalEncodingLayer
%load_ext autoreload
%autoreload 2

c:\Users\user\miniconda3\envs\tf_py3.10\python.exe


In [ ]:
from urllib.request import urlretrieve
urlretrieve("http://images.cocodataset.org/zips/train2017.zip", "train2017.zip")
urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "coco.zip")

('coco.zip', <http.client.HTTPMessage at 0x2b788ba3520>)

In [ ]:
import zipfile
zipfile.ZipFile('train2017.zip', 'r').extractall('train2017')
zipfile.ZipFile('coco.zip', 'r').extractall('coco')

In [2]:
import json
train_captions = json.load(open("coco/annotations/captions_train2017.json", 'r'))

In [3]:
import cv2
image_data = []
image_data_id = []
for image in train_captions['images'][:1000]:
    img = cv2.imread(f"train2017/train2017/{image['file_name']}")
    img = cv2.resize(img, (572, 572))
    image_data.append(img)
    image_data_id.append(image["id"])

In [4]:
import numpy as np
image_data = np.array(image_data)
image_data_id = np.array(image_data_id)

In [5]:
caption_data = []
caption_data_image_id =[]
for annotation in train_captions['annotations']:
    if (image_data_id == annotation['image_id']).sum() == 1:
        caption_data.append(annotation['caption'])
        caption_data_image_id.append(annotation['image_id'])
caption_data = np.array(caption_data)
caption_data_image_id = np.array(caption_data_image_id)

In [6]:
chars = set()
for caption in caption_data:
    for char in caption:
        chars.add(char)

In [7]:
chars.add('\0')
chars_arr = np.array(list(chars))

In [8]:
num_tokens = np.array([len(caption) for caption in caption_data]).max()

In [9]:
print(num_tokens)
num_tokens=192

185


In [10]:
tokenized = []
for caption in caption_data:
    seq = []
    for i in range(num_tokens):
        try:
            seq.append(chars_arr == caption[i])
        except IndexError:
            seq.append(chars_arr == '\0')
    tokenized.append(seq)

In [11]:
tokenized_captions = np.array(tokenized)

In [12]:
caption_matching_images = []
for id in caption_data_image_id:
    caption_matching_images.append(image_data[image_data_id==id])
caption_matching_images = np.array(caption_matching_images)

In [13]:
np.where(chars_arr == '\0')

(array([24], dtype=int64),)

In [14]:
print(tokenized_captions.shape)

(5004, 192, 72)


In [16]:
mask =  tf.where(tokenized_captions[:, :, np.where(chars_arr == '\0')], -1e9, 0.0) 

In [34]:
import gc
gc.collect()

0

In [22]:
A = DensePCNLayer(128, 0.00001)
C = DensePCNLayer(200, 0.00001)
A.state = tf.Variable(tf.random.normal((1, 192, 128)))
B = TransformerPCNLayer(3, 128, 8, 0.00001, A, [C], mask[0][None])
A.next_layers=[B]
C.prev_layer = B
b_out = B(A.state)
c_out = C(b_out)

In [23]:
for i in range(10):
    for layer in B.get_layers():
        layer.update_state()
        layer.update_wts()
        layer.update_b()
    print(tf.reduce_sum((B.predict_next() - B(A.predict_next()))**2)
        +tf.reduce_sum((C.predict_next() - C(B.predict_next()))**2)
        +tf.reduce_sum((A.predict_next()-B.predict_prev())**2)
        +tf.reduce_sum((B.predict_next()-C.predict_prev())**2))

tf.Tensor(225544.53, shape=(), dtype=float32)
tf.Tensor(222998.22, shape=(), dtype=float32)
tf.Tensor(220503.75, shape=(), dtype=float32)
tf.Tensor(218059.17, shape=(), dtype=float32)
tf.Tensor(215662.56, shape=(), dtype=float32)
tf.Tensor(213312.34, shape=(), dtype=float32)
tf.Tensor(211006.98, shape=(), dtype=float32)
tf.Tensor(208745.19, shape=(), dtype=float32)
tf.Tensor(206525.73, shape=(), dtype=float32)
tf.Tensor(204347.45, shape=(), dtype=float32)


In [28]:
print(tokenized_captions.shape)

(5004, 192, 72)


In [35]:
from encoder_encoder_pcn import EncoderEncoderPCN
eepcn = EncoderEncoderPCN(1e-5)
eepcn.pass_through(tf.cast(caption_matching_images[0], tf.float32), tf.cast(tf.convert_to_tensor(tokenized_captions[0][None]), tf.float32) )

ResourceExhaustedError: {{function_node __wrapped__Relu_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:Relu]